# NFL Coaching Data Collection Add-On
## Purpose
This notebook adds coaching history, team-season context, coaching changes, and play-calling tendency features to the NFL public-data warehouse. Run it after `NFL_Public_Data_Warehouse.ipynb`, or change `DATA_ROOT` to point to the directory where that notebook exported its datasets.

**Important:** Public sources vary in coverage and license. The notebook does not pretend that every coordinator tenure or scheme label is available historically. Unknown values remain missing; every dataset records source and retrieval metadata.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import json, re, warnings
import pandas as pd
import numpy as np

DATA_ROOT = Path("nfl_data")       # change to your existing warehouse export directory
COACH_ROOT = DATA_ROOT / "coaching"
COACH_ROOT.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH = COACH_ROOT / "coaching_source_manifest.csv"
RUN_TS = datetime.now(timezone.utc).isoformat()

def save_table(df, name):
    # Parquet preferred; CSV is a portable fallback.
    path = COACH_ROOT / f"{name}.parquet"
    try:
        df.to_parquet(path, index=False)
        fmt = "parquet"
    except Exception:
        path = COACH_ROOT / f"{name}.csv"
        df.to_csv(path, index=False)
        fmt = "csv"
    print(f"{name}: {len(df):,} rows -> {path} ({fmt})")
    return str(path)

## Source catalog

Start with sources that can be accessed without scraping behind logins. For each source, record its actual URL, coverage, terms, and whether a download/API was successful. `nflverse` play-by-play and schedules support calculated tendencies; public coach-history pages and repositories can supply tenure data, but should be independently verified before modeling.

In [2]:
coaching_sources = pd.DataFrame([
    {"source":"nflverse / nflreadpy", "purpose":"Play-by-play, schedules, team/player IDs; calculate tendencies", "url":"https://nflreadr.nflverse.com/ ; https://github.com/nflverse/nflverse-data", "access":"Python package / public data", "limitations":"Coverage and columns vary by dataset and season; verify current license"},
    {"source":"Pro Football Reference", "purpose":"Head-coach and team season history; cross-check results", "url":"https://www.pro-football-reference.com/", "access":"Public website", "limitations":"Automated access may be restricted; do not bypass access controls"},
    {"source":"NFL.com", "purpose":"Official team/coach pages and stats", "url":"https://www.nfl.com/", "access":"Public website", "limitations":"Historical coverage and structured access vary"},
    {"source":"ESPN", "purpose":"Coach/team news and event context", "url":"https://www.espn.com/nfl/", "access":"Public website / endpoints may change", "limitations":"Unstable undocumented endpoints; validate data"},
    {"source":"Wikipedia / Wikidata", "purpose":"Candidate coach tenure timeline and identity reconciliation", "url":"https://www.wikidata.org/ ; https://en.wikipedia.org/wiki/Category:National_Football_League_coaches", "access":"Public structured/community data", "limitations":"Use as a lead, verify against stronger sources"},
    {"source":"GitHub public repositories", "purpose":"Community-maintained coach/team mappings", "url":"https://github.com/search?q=NFL+coaching+history&type=repositories", "access":"Public repository", "limitations":"Inspect provenance, license, and freshness"},
], columns=["source","purpose","url","access","limitations"])
display(coaching_sources)

,source,purpose,url,access,limitations
0,nflverse / nflreadpy,"Play-by-play, schedules, team/player IDs; calc...",https://nflreadr.nflverse.com/ ; https://githu...,Python package / public data,Coverage and columns vary by dataset and seaso...
1,Pro Football Reference,Head-coach and team season history; cross-chec...,https://www.pro-football-reference.com/,Public website,Automated access may be restricted; do not byp...
2,NFL.com,Official team/coach pages and stats,https://www.nfl.com/,Public website,Historical coverage and structured access vary
3,ESPN,Coach/team news and event context,https://www.espn.com/nfl/,Public website / endpoints may change,Unstable undocumented endpoints; validate data
4,Wikipedia / Wikidata,Candidate coach tenure timeline and identity r...,https://www.wikidata.org/ ; https://en.wikiped...,Public structured/community data,"Use as a lead, verify against stronger sources"
5,GitHub public repositories,Community-maintained coach/team mappings,https://github.com/search?q=NFL+coaching+histo...,Public repository,"Inspect provenance, license, and freshness"


In [3]:
# Preserve a source registry even before every source is ingested.
manifest = coaching_sources.copy()
manifest["retrieved_at_utc"] = RUN_TS
manifest["status"] = "cataloged_not_yet_ingested"
manifest["dataset"] = ""
manifest["rows"] = pd.NA
manifest["notes"] = ""
manifest.to_csv(MANIFEST_PATH, index=False)
print("Wrote source manifest:", MANIFEST_PATH)

Wrote source manifest: nfl_data\coaching\coaching_source_manifest.csv


## Load existing warehouse tables

The helper tries Parquet first, then CSV. It searches recursively so that the coaching add-on can use the existing collection notebook's output layout.

In [4]:
def load_existing(stem):
    candidates = list(DATA_ROOT.rglob(f"{stem}.parquet")) + list(DATA_ROOT.rglob(f"{stem}.csv"))
    if not candidates:
        return pd.DataFrame()
    p = candidates[0]
    try:
        df = pd.read_parquet(p) if p.suffix == ".parquet" else pd.read_csv(p, low_memory=False)
        print(f"Loaded {stem}: {len(df):,} rows from {p}")
        return df
    except Exception as e:
        print(f"Could not read {p}: {e}")
        return pd.DataFrame()

pbp = load_existing("pbp")
schedules = load_existing("schedules")
team_stats = load_existing("team_stats")

Loaded pbp: 1,293,659 rows from nfl_data\pbp.parquet
Loaded schedules: 7,548 rows from nfl_data\schedules.parquet
Loaded team_stats: 14,691 rows from nfl_data\team_stats.parquet


## Build play-calling tendency tables from play-by-play

Column names can change by release, so the code detects common nflverse names. It creates team-game and team-season aggregates only when the required fields exist. Rates are descriptive proxies, not a definitive measure of a coach's intent. Filter garbage time or use situation-specific subsets for stronger analyses.

In [5]:
def first_col(df, options):
    return next((c for c in options if c in df.columns), None)

tendency_game = pd.DataFrame()
tendency_season = pd.DataFrame()
if not pbp.empty:
    season_col = first_col(pbp, ["season"])
    week_col = first_col(pbp, ["week"])
    game_col = first_col(pbp, ["game_id"])
    team_col = first_col(pbp, ["posteam", "offense_team", "team"])
    play_col = first_col(pbp, ["play_type"])
    pass_col = first_col(pbp, ["pass_attempt"])
    rush_col = first_col(pbp, ["rush_attempt"])
    success_col = first_col(pbp, ["success"])
    epa_col = first_col(pbp, ["epa"])
    if all([season_col, team_col]) and (play_col or pass_col or rush_col):
        x = pbp.copy()
        x["_team"] = x[team_col]
        x["_season"] = x[season_col]
        if week_col: x["_week"] = x[week_col]
        if game_col: x["_game"] = x[game_col]
        if play_col:
            ptype = x[play_col].astype(str).str.lower()
            x["_pass"] = ptype.eq("pass").astype(int)
            x["_rush"] = ptype.eq("run").astype(int)
        else:
            x["_pass"] = pd.to_numeric(x[pass_col], errors="coerce").fillna(0) if pass_col else 0
            x["_rush"] = pd.to_numeric(x[rush_col], errors="coerce").fillna(0) if rush_col else 0
        x["_play"] = ((x["_pass"] + x["_rush"]) > 0).astype(int)
        if success_col: x["_success"] = pd.to_numeric(x[success_col], errors="coerce")
        if epa_col: x["_epa"] = pd.to_numeric(x[epa_col], errors="coerce")
        keys = ["_season", "_team"] + (["_week", "_game"] if week_col and game_col else [])
        aggs = {"plays":("_play","sum"),"pass_plays":("_pass","sum"),"rush_plays":("_rush","sum")}
        if success_col: aggs["success_rate"]=("_success","mean")
        if epa_col: aggs["mean_epa"]=("_epa","mean")
        tendency_game = x.groupby(keys, dropna=False).agg(**aggs).reset_index()
        denom = tendency_game["plays"].replace(0, np.nan)
        tendency_game["pass_rate"] = tendency_game["pass_plays"] / denom
        tendency_game["rush_rate"] = tendency_game["rush_plays"] / denom
        tendency_game = tendency_game.rename(columns={"_season":"season","_team":"team","_week":"week","_game":"game_id"})
        season_keys = ["season","team"]
        tendency_season = tendency_game.groupby(season_keys, dropna=False).agg(
            games=("plays","count"), plays=("plays","sum"), pass_plays=("pass_plays","sum"),
            rush_plays=("rush_plays","sum"), mean_pass_rate=("pass_rate","mean"),
            mean_rush_rate=("rush_rate","mean")
        ).reset_index()
        tendency_season["pass_rate_weighted"] = tendency_season["pass_plays"] / tendency_season["plays"].replace(0, np.nan)
        tendency_season["rush_rate_weighted"] = tendency_season["rush_plays"] / tendency_season["plays"].replace(0, np.nan)
    else:
        print("Could not detect the minimum season/team/play fields. Inspect pbp.columns and adjust mappings.")
else:
    print("No pbp table found. Set DATA_ROOT correctly or run the main collector first.")

display(tendency_game.head())
display(tendency_season.head())

,season,team,week,game_id,plays,pass_plays,rush_plays,success_rate,mean_epa,pass_rate,rush_rate
0,1999,,1,1999_01_CIN_TEN,0,0,0,0.0,-5.067937,NaN,NaN
1,1999,,1,1999_01_KC_CHI,0,0,0,0.0,-4.901083,NaN,NaN
2,1999,,2,1999_02_ATL_DAL,0,0,0,0.0,0.000000,NaN,NaN
3,1999,,2,1999_02_DEN_KC,0,0,0,0.0,-5.391541,NaN,NaN
4,1999,,2,1999_02_IND_NE,0,0,0,0.0,-8.566886,NaN,NaN


,season,team,games,plays,pass_plays,rush_plays,mean_pass_rate,mean_rush_rate,pass_rate_weighted,rush_rate_weighted
0,1999,,32,0,0,0,NaN,NaN,NaN,NaN
1,1999,ARI,16,983,599,384,0.608534,0.391466,0.609359,0.390641
2,1999,ATL,16,914,546,368,0.598452,0.401548,0.597374,0.402626
3,1999,BAL,15,956,553,403,0.576050,0.423950,0.578452,0.421548
4,1999,BUF,17,1081,560,521,0.519976,0.480024,0.518039,0.481961


## Coach identity and tenure tables

These are schema templates. Populate them from verified public sources, keeping one row per coach-role-team interval. Do not infer coordinator tenure from team results alone.

In [6]:
coach_columns = ["coach_id","coach_name","role","team","start_season","end_season","interim","source_url","source_checked_at"]
coaches = pd.DataFrame(columns=["coach_id","coach_name","birth_year","source_url"])
coach_tenures = pd.DataFrame(columns=coach_columns)
coaching_changes = pd.DataFrame(columns=["change_date","season","team","role","coach_out","coach_in","change_type","source_url"])
# Optional manual seed: add verified rows to these DataFrames, or import your own CSV.
# Example:
# coach_tenures.loc[len(coach_tenures)] = ["coach_001","Example Name","head_coach","KC",2024,2025,False,"https://...",RUN_TS]
display(coaches)
display(coach_tenures)

,coach_id,coach_name,birth_year,source_url


,coach_id,coach_name,role,team,start_season,end_season,interim,source_url,source_checked_at


In [7]:
# Export each table independently. Empty schema templates are saved too, making missing coverage explicit.
tables = {
    "coaching_sources": coaching_sources,
    "coaches": coaches,
    "coach_tenures": coach_tenures,
    "coaching_changes": coaching_changes,
    "team_game_tendencies": tendency_game,
    "team_season_tendencies": tendency_season,
}
for name, df in tables.items():
    save_table(df, name)

print("\nCoverage reminder: coach history and coordinator assignments remain incomplete until verified source rows are added.")

coaching_sources: 6 rows -> nfl_data\coaching\coaching_sources.parquet (parquet)
coaches: 0 rows -> nfl_data\coaching\coaches.parquet (parquet)
coach_tenures: 0 rows -> nfl_data\coaching\coach_tenures.parquet (parquet)
coaching_changes: 0 rows -> nfl_data\coaching\coaching_changes.parquet (parquet)
team_game_tendencies: 22,125 rows -> nfl_data\coaching\team_game_tendencies.parquet (parquet)
team_season_tendencies: 923 rows -> nfl_data\coaching\team_season_tendencies.parquet (parquet)

Coverage reminder: coach history and coordinator assignments remain incomplete until verified source rows are added.


## Suggested next step
Use `team_game_tendencies` as historical features in the analysis notebook. Join coach tenure using the game date/season and team, never using a coach hired after the target game. For predictive models, calculate rolling tendencies using only games completed before each prediction cutoff.